In [1]:
import dspy

# Konfiguration des lokalen Sprachmodells
local_llm = dspy.LM(
    "openai/gemma-3-4b-it-Q4_K_M.gguf", 
    api_base="http://localhost:8080/v1", 
    api_key="no_key_needed",
    temperature=0.1,
    cache=False
)

dspy.configure(lm=local_llm)

In [2]:
import heapq
from datasets import Dataset, load_dataset

# Lade die kürzesten Dokumente, da die längeren Dokumente nicht in das 128k Kontext Fenster des LLM passen
# Aber der Artikel muss mindestesn 10.000 Zeichen lang sein, da in dem Dataset doch einiges an Schrott drin ist :-(
def shortest_n(dataset, n, min_length=10000):
    """
    Wählt die n kürzesten Artikel aus, die mindestens `min_length` Zeichen lang sind.
    """
    # Filter auf Mindestlänge anwenden
    filtered = (x for x in dataset if len(x["article"]) >= min_length)
    
    # n kürzeste behalten
    return heapq.nsmallest(n, filtered, key=lambda x: len(x["article"]))

dataset = load_dataset("ccdv/arxiv-summarization", split="train")
shortest_125 = shortest_n(dataset, 125)
shortest_dataset = Dataset.from_list(shortest_125)

In [3]:
train_samples = shortest_dataset.select(range(100))
dev_samples = shortest_dataset.select(range(100, 125))

trainset = [dspy.Example(long_text=ex['article'], summary=ex['abstract']).with_inputs('long_text') for ex in train_samples]
devset = [dspy.Example(long_text=ex['article'], summary=ex['abstract']).with_inputs('long_text') for ex in dev_samples]

print(f"Anzahl der Beispiele im Trainingsset: {len(trainset)}")
print(f"Anzahl der Beispiele im Evaluationsset: {len(devset)}")

Anzahl der Beispiele im Trainingsset: 100
Anzahl der Beispiele im Evaluationsset: 25


In [4]:
# Diese mal sind die Sgnaturen auf englich, da das Dataset auch auf englisch ist und 
# die rouge_metric nur funktioniert, wenn wie Sprache von gold unf pred identisch ist.
class GenerateKeywords(dspy.Signature):
    """Generates a list of 10 relevant keywords from a long scientific text."""
    long_text = dspy.InputField(desc="A long scientific article from which keywords should be extracted.")
    keywords = dspy.OutputField(desc="A comma-separated list of 5–7 keywords.")

class SummarizeWithKeywords(dspy.Signature):
    """Creates a short summary of a scientific text while taking the provided keywords into account."""
    long_text = dspy.InputField(desc="The original scientific text.")
    keywords = dspy.InputField(desc="Relevant keywords that the summary should focus on.")
    summary = dspy.OutputField(desc="A concise summary of the article in 3–4 sentences.")

class SummarizationModule(dspy.Module):
    def __init__(self):
        super().__init__()
        self.keyword_generator = dspy.Predict(GenerateKeywords)
        self.summarizer = dspy.Predict(SummarizeWithKeywords)

    def forward(self, long_text):
        keywords_prediction = self.keyword_generator(long_text=long_text)
        summary_prediction = self.summarizer(long_text=long_text, keywords=keywords_prediction.keywords)
        return dspy.Prediction(summary=summary_prediction.summary)

In [5]:
# ROUGE-L ist eine Standardmetrik zur Bewertung von Zusammenfassungen.
import evaluate as hf_evaluate

rouge = hf_evaluate.load('rouge')

def rouge_metric(gold, pred, trace=None):
    """Berechnet den ROUGE-L F1-Score."""
    if not pred.summary or not gold.summary:
        return 0.0
    results = rouge.compute(predictions=[pred.summary], references=[gold.summary])
    return results['rougeL']

In [6]:
# Der Optimizer generiert für jeden Prädiktor im Modul bis zu 3 "Few-Shot"-Beispiele.
from dspy.teleprompt import BootstrapFewShot

optimizer = BootstrapFewShot(metric=rouge_metric, max_bootstrapped_demos=3)

# Der compile-Prozess wendet den Optimizer auf das Modul an.
print("\nBeginne die Kompilierung des Moduls. Dies kann einige Minuten dauern...")
compiled_summarizer = optimizer.compile(SummarizationModule(), trainset=trainset)
print("Kompilierung abgeschlossen.")


Beginne die Kompilierung des Moduls. Dies kann einige Minuten dauern...


  3%|███▍                                                                                                                | 3/100 [02:00<1:05:02, 40.24s/it]

Bootstrapped 3 full traces after 3 examples for up to 1 rounds, amounting to 3 attempts.
Kompilierung abgeschlossen.


In [7]:
# --- 6. Evaluation des unoptimierten und optimierten Moduls ---
from dspy.evaluate import Evaluate

evaluator = Evaluate(devset=devset, num_threads=1, display_progress=True, display_table=5)

print("\n--- Evaluation des unoptimierten Moduls (Zero-Shot) ---")
unoptimized_summarizer = SummarizationModule()
evaluator(unoptimized_summarizer, metric=rouge_metric)

print("\n--- Evaluation des optimierten Moduls (Few-Shot) ---")
evaluator(compiled_summarizer, metric=rouge_metric)


--- Evaluation des unoptimierten Moduls (Zero-Shot) ---
Average Metric: 5.15 / 25 (20.6%): 100%|███████████████████████████████████████████████████████████████████████████████████| 25/25 [03:31<00:00,  8.47s/it]

2025/11/13 19:34:53 INFO dspy.evaluate.evaluate: Average Metric: 5.154867295756456 / 25 (20.6%)


,long_text,example_summary,pred_summary,rouge_metric
0,"in the large scale region of wealth , income , profits , assets , ...",we investigate the dynamical behavior in the large scale region of...,This research investigates the dynamics of wealth distribution in ...,✔️ [0.234]
1,collective states constitute a significant part of the spectra of ...,the shell model monte carlo ( smmc ) approach allows for the micro...,This article discusses recent applications of the Shell Model Mont...,✔️ [0.287]
2,providing a repeatable movement is essential for a wide range of w...,when deploying a testbed infrastructure for wireless sensor networ...,Gondola is a novel robotic infrastructure designed to overcome lim...,✔️ [0.137]
3,"over the past decade , relativistic hydrodynamics has established ...",we show that in hydrodynamic simulations for relativistic heavy - ...,This research focuses on optimizing the hydro-to-hadron conversion...,✔️ [0.160]
4,the observations of mrk 501 in the very high energy ( ) domain wer...,the blazar markarian 501 ( mrk 501 ) was observed above 100 with t...,This study investigated the temporal and spectral variability of t...,✔️ [0.221]



--- Evaluation des optimierten Moduls (Few-Shot) ---
Average Metric: 5.56 / 25 (22.3%): 100%|███████████████████████████████████████████████████████████████████████████████████| 25/25 [34:11<00:00, 82.05s/it]

2025/11/13 20:09:04 INFO dspy.evaluate.evaluate: Average Metric: 5.562890051371735 / 25 (22.3%)


,long_text,example_summary,pred_summary,rouge_metric
0,"in the large scale region of wealth , income , profits , assets , ...",we investigate the dynamical behavior in the large scale region of...,this study investigates the dynamical behavior of non-equilibrium ...,✔️ [0.280]
1,collective states constitute a significant part of the spectra of ...,the shell model monte carlo ( smmc ) approach allows for the micro...,the shell model monte carlo ( smmc ) method has been successfully ...,✔️ [0.274]
2,providing a repeatable movement is essential for a wide range of w...,when deploying a testbed infrastructure for wireless sensor networ...,"we present gondola, a novel robotic infrastructure for wireless se...",✔️ [0.176]
3,"over the past decade , relativistic hydrodynamics has established ...",we show that in hydrodynamic simulations for relativistic heavy - ...,this paper outlines a strategy to improve the accuracy of anisotro...,✔️ [0.163]
4,the observations of mrk 501 in the very high energy ( ) domain wer...,the blazar markarian 501 ( mrk 501 ) was observed above 100 with t...,during the observation campaign of mrk 501 between may and july 20...,✔️ [0.205]


EvaluationResult(score=22.25, results=<list of 25 results>)

In [8]:
# --- 7. Inspektion der optimierten Prompts ---
print("\n--- Inspektion eines optimierten Prompts ---")
# Ein Beispieldurchlauf, um die internen Prompts zu inspizieren.
example_dev_text = devset[0].long_text
compiled_summarizer(long_text=example_dev_text)


--- Inspektion eines optimierten Prompts ---


Prediction(
    summary='this study investigates the dynamical behavior of non-equilibrium systems in the large scale region of wealth using data on assessed land values in Japan from 1983 to 2006. \n it identifies a detailed quasi-balance, derived from gibrat’s law and the detailed balance, which allows for the derivation of a varying pareto index over time. \n the analysis reveals a significant shift in the pareto index between 1984 and 1986, indicating a period of rapid change in the system’s dynamics, and highlights the differences between the large and middle scale regions of economic distribution.'
)

In [9]:
# Inspektion des letzten Prompts, der an den 'summarizer' Prädiktor gesendet wurde.
local_llm.inspect_history(n=1)





[2025-11-13T20:17:28.665959]

System message:

Your input fields are:
1. `long_text` (str): The original scientific text.
2. `keywords` (str): Relevant keywords that the summary should focus on.
Your output fields are:
1. `summary` (str): A concise summary of the article in 3–4 sentences.
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## long_text ## ]]
{long_text}

[[ ## keywords ## ]]
{keywords}

[[ ## summary ## ]]
{summary}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Creates a short summary of a scientific text while taking the provided keywords into account.


User message:

This is an example of the task, though some input or output fields are not supplied.

[[ ## long_text ## ]]
the study of domain walls ( dw ) magnetic ( static and dynamic ) @xcite and transport @xcite properties have attracted much attention recently due to their relevance for magneto - electronic nano - device appl